In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.ticker as plticker

from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [3]:
world_cup = pd.read_csv('/content/odi mens  team ranking 2026.csv')
results = pd.read_csv('/content/wc_dataset.csv')

In [ ]:
world_cup.head()

,Rank,Team,Matches,Points,Rating
0,1,India,45,5377,119
1,2,New Zealand,50,5559,111
2,3,Australia,38,4134,109
3,4,South Africa,44,4475,102
4,5,Pakistan,41,4022,98


In [4]:
results.head()

,Team 1,Team 2,Winner,Win Margin,Ground,Match Date
0,New Zealand,England,New Zealand,11 runs,"Gujarat Stadium, Motera, Ahmedabad",1996-02-14
1,South Africa,United Arab Emirates,South Africa,169 runs,Rawalpindi Cricket Stadium,1996-02-16
2,Zimbabwe,West Indies,West Indies,6 wickets,"Lal Bahadur Shastri Stadium, Hyderabad, Deccan",1996-02-16
3,New Zealand,Netherlands,New Zealand,119 runs,Indian Petrochemicals Corporation Limited Spor...,1996-02-17
4,Sri Lanka,Australia,Sri Lanka,Walkover,"R Premadasa Stadium, Colombo",1996-02-17


In [7]:
#filtering the matches playes by australia
df = results[(results['Team 1'] == 'Australia') | (results['Team 2'] == 'Australia')]
Australia = df.iloc[:]
Australia.head()

,Team 1,Team 2,Winner,Win Margin,Ground,Match Date
4,Sri Lanka,Australia,Sri Lanka,Walkover,"R Premadasa Stadium, Colombo",1996-02-17
11,Australia,Kenya,Australia,97 runs,"Indira Priyadarshini Stadium, Visakhapatnam",1996-02-23
18,Australia,India,Australia,16 runs,"Wankhede Stadium, Mumbai",1996-02-27
21,Zimbabwe,Australia,Australia,8 wickets,"Vidarbha Cricket Association Ground, Nagpur",1996-03-01
25,Australia,West Indies,West Indies,4 wickets,"Sawai Mansingh Stadium, Jaipur",1996-03-04


In [35]:


# Remove extra spaces from column names
Australia.columns = Australia.columns.str.strip()

# Convert Match Date to datetime
Australia["Match Date"] = pd.to_datetime(Australia["Match Date"], errors="coerce")

# Create Match Year column
Australia["Match Year"] = Australia["Match Date"].dt.year

# Keep matches from 2023 onwards
Australia_2023 = Australia[Australia["Match Year"] <= 2023]

# Display result
print(Australia_2023.shape)
print(Australia_2023.head())

(55, 7)
       Team 1       Team 2       Winner Win Margin  \
4   Sri Lanka    Australia    Sri Lanka   Walkover   
11  Australia        Kenya    Australia    97 runs   
18  Australia        India    Australia    16 runs   
21   Zimbabwe    Australia    Australia  8 wickets   
25  Australia  West Indies  West Indies  4 wickets   

                                         Ground Match Date  Match Year  
4                  R Premadasa Stadium, Colombo 1996-02-17        1996  
11  Indira Priyadarshini Stadium, Visakhapatnam 1996-02-23        1996  
18                     Wankhede Stadium, Mumbai 1996-02-27        1996  
21  Vidarbha Cricket Association Ground, Nagpur 1996-03-01        1996  
25               Sawai Mansingh Stadium, Jaipur 1996-03-04        1996  


In [36]:
worldcup_teams = ['England', 'South Africa', 'West Indies', 'Pakistan', 'New Zealand', 'Sri Lanka', 'Afghanistan', 'Australia', 'Bangladesh', 'India']
df_teams_1 = results[results['Team 1'].isin(worldcup_teams)]
df_teams_2 = results[results['Team 2'].isin(worldcup_teams)]
df_teams = pd.concat((df_teams_1, df_teams_2))
df_teams.drop_duplicates()
df_teams.count()

,0
Team 1,455
Team 2,455
Winner,455
Win Margin,437
Ground,455
Match Date,455


In [37]:
#drop the column that will not effect the match outcomes
df_teams_2023 = df_teams.drop(['Ground','Match Date','Win Margin'], axis=1)
df_teams_2023.head()

,Team 1,Team 2,Winner
0,New Zealand,England,New Zealand
1,South Africa,United Arab Emirates,South Africa
3,New Zealand,Netherlands,New Zealand
4,Sri Lanka,Australia,Sri Lanka
7,New Zealand,South Africa,South Africa


In [38]:
# building the model
# the prediction label : The winning team column will show '1' Team 1 has won and '2' if the away team has won
df_teams_2023 = df_teams_2023.reset_index(drop=True)
df_teams_2023.loc[df_teams_2023.Winner == df_teams_2023['Team 1'],'winning_team']=1
df_teams_2023.loc[df_teams_2023.Winner == df_teams_2023['Team 2'],'winning_team']=2
df_teams_2023 = df_teams_2023.drop(['winning_team'], axis=1)


In [39]:
#displaying the teams won in 2019
df_teams_2023.head()


,Team 1,Team 2,Winner
0,New Zealand,England,New Zealand
1,South Africa,United Arab Emirates,South Africa
2,New Zealand,Netherlands,New Zealand
3,Sri Lanka,Australia,Sri Lanka
4,New Zealand,South Africa,South Africa


In [40]:
#converting team 1 and 2 into categorical values
final = pd.get_dummies(df_teams_2023, prefix=['Team_1', 'Team_2'], columns=['Team 1', 'Team 2'])

#seperate x and y sets
x = final.drop(['Winner'], axis=1)
y = final["Winner"]

#splitting the data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.30, random_state = 42)

In [41]:
final.head()

,Winner,Team_1_Afghanistan,Team_1_Australia,Team_1_Bangladesh,Team_1_Bermuda,Team_1_Canada,Team_1_East Africa,Team_1_England,Team_1_India,Team_1_Ireland,...,Team_2_Namibia,Team_2_Netherlands,Team_2_New Zealand,Team_2_Pakistan,Team_2_Scotland,Team_2_South Africa,Team_2_Sri Lanka,Team_2_United Arab Emirates,Team_2_West Indies,Team_2_Zimbabwe
0,New Zealand,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,South Africa,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,New Zealand,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
3,Sri Lanka,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,South Africa,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False


In [42]:
#training the model using svm algorithm

from sklearn import svm
model = svm.SVC()

svm = svm.SVC(gamma = 'scale')
svm.fit(x_train, y_train)

score = svm.score(x_train, y_train)
score2 = svm.score(x_test, y_test)

print("Training set accuracy: ", '%.3f'%(score))
print("Test set accuracy: ", '%.3f'%(score2))


Training set accuracy:  0.808
Test set accuracy:  0.693


In [43]:
#Adding ICC ranking
ranking = pd.read_csv('/content/odi mens  team ranking 2026.csv')
fixtures = pd.read_csv('/content/dummy fixtures.csv')


pred_set =[]

In [44]:
#create new colums with ranking of each team
fixtures.insert(1, 'first_position', fixtures['Team1'].map(ranking.set_index('Team')['Rank']))
fixtures.insert(2, 'second_position', fixtures['Team2'].map(ranking.set_index('Team')['Rank']))


fixtures = fixtures.iloc[:45, :1]
fixtures.tail()

,Match_No
21,22
22,23
23,24
24,25
25,26


In [45]:
fixtures = pd.read_csv("/content/dummy fixtures.csv")

In [46]:
print(fixtures.head())
print(fixtures.columns)

   Match_No        Date Group         Team1        Team2         Venue  \
0         1  2027-10-10     A         India     Zimbabwe  Johannesburg   
1         2  2027-10-10     B     Australia  Netherlands     Cape Town   
2         3  2027-10-11     A      Pakistan   Bangladesh      Pretoria   
3         4  2027-10-11     B       England      Namibia      Windhoek   
4         5  2027-10-12     A  South Africa  Afghanistan        Durban   

   Host_Country  
0  South Africa  
1  South Africa  
2  South Africa  
3       Namibia  
4  South Africa  
Index(['Match_No', 'Date', 'Group', 'Team1', 'Team2', 'Venue', 'Host_Country'], dtype='object')


In [47]:
rank_map = dict(zip(ranking["Team"].str.strip(), ranking["Rank"]))

fixtures["first_position"] = fixtures["Team1"].map(rank_map)
fixtures["second_position"] = fixtures["Team2"].map(rank_map)

In [48]:
print(fixtures[fixtures["first_position"].isna()][["Team1"]].drop_duplicates())
print(fixtures[fixtures["second_position"].isna()][["Team2"]].drop_duplicates())

          Team1
17      Namibia
18     Zimbabwe
23  Netherlands
         Team2
0     Zimbabwe
1  Netherlands
3      Namibia


In [49]:
fixtures["rank_diff"] = abs(
    fixtures["first_position"] - fixtures["second_position"]
)

In [50]:
fixtures["home_advantage"] = (fixtures["first_position"] < fixtures["second_position"]).astype(int)

In [51]:
# Step 1: clean column names
fixtures.columns = fixtures.columns.str.strip()
ranking.columns = ranking.columns.str.strip()

# Step 2: make sure ranking columns are clean
ranking["Team"] = ranking["Team"].str.strip()

# Step 3: create rank dictionary
rank_map = dict(zip(ranking["Team"], ranking["Rank"]))

# Step 4: map rankings to fixtures (NO insert needed)
fixtures["first_position"] = fixtures["Team1"].str.strip().map(rank_map)
fixtures["second_position"] = fixtures["Team2"].str.strip().map(rank_map)

# Step 5: keep only first 45 rows (optional)
fixtures = fixtures.iloc[:45]

# Step 6: view result
fixtures.head()

,Match_No,Date,Group,Team1,Team2,Venue,Host_Country,first_position,second_position,rank_diff,home_advantage
0,1,2027-10-10,A,India,Zimbabwe,Johannesburg,South Africa,1.0,NaN,NaN,0
1,2,2027-10-10,B,Australia,Netherlands,Cape Town,South Africa,3.0,NaN,NaN,0
2,3,2027-10-11,A,Pakistan,Bangladesh,Pretoria,South Africa,5.0,9.0,4.0,1
3,4,2027-10-11,B,England,Namibia,Windhoek,Namibia,8.0,NaN,NaN,0
4,5,2027-10-12,A,South Africa,Afghanistan,Durban,South Africa,4.0,7.0,3.0,1


In [52]:
#loop to add teams to new prediction dataset based on the ranking position at each time

for index, row in fixtures.iterrows():
  if row['first_position'] < row['second_position']:
    pred_set.append({'Team1': row['Team1'], 'Team2': row['Team2'], 'winning_team': None})
  else:
    pred_set.append({'Team1': row['Team2'], 'Team2': row['Team1'], 'winning_team': None})

pred_set = pd.DataFrame(pred_set)
backup_pred_set = pred_set
pred_set.head()

,Team1,Team2,winning_team
0,Zimbabwe,India,None
1,Netherlands,Australia,None
2,Pakistan,Bangladesh,None
3,Namibia,England,None
4,South Africa,Afghanistan,None


In [53]:
pred_set.columns = pred_set.columns.str.strip()

pred_set = pd.get_dummies(
    pred_set,
    prefix=["Team1", "Team2"],
    columns=["Team1", "Team2"]
)

# align columns with training set
missing_cols = set(final.columns) - set(pred_set.columns)
for c in missing_cols:
    pred_set[c] = 0

# keep same order as training data
pred_set = pred_set[final.columns]

# drop target column if present
if "Winner" in pred_set.columns:
    pred_set = pred_set.drop(["Winner"], axis=1)

pred_set.head()

,Team_1_Afghanistan,Team_1_Australia,Team_1_Bangladesh,Team_1_Bermuda,Team_1_Canada,Team_1_East Africa,Team_1_England,Team_1_India,Team_1_Ireland,Team_1_Kenya,...,Team_2_Namibia,Team_2_Netherlands,Team_2_New Zealand,Team_2_Pakistan,Team_2_Scotland,Team_2_South Africa,Team_2_Sri Lanka,Team_2_United Arab Emirates,Team_2_West Indies,Team_2_Zimbabwe
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [54]:
#group matches
predictions = svm.predict(pred_set)
for i in range(fixtures.shape[0]):
  print(backup_pred_set.iloc[i, 1] + " and " + backup_pred_set.iloc[i, 0])
  if predictions[i] == 1:
    print("Winner: " + backup_pred_set.iloc[i, 1])
  else:
    print("Winner: " + backup_pred_set.iloc[i, 0])
  print("")

India and Zimbabwe
Winner: Zimbabwe

Australia and Netherlands
Winner: Netherlands

Bangladesh and Pakistan
Winner: Pakistan

England and Namibia
Winner: Namibia

Afghanistan and South Africa
Winner: South Africa

Sri Lanka and New Zealand
Winner: New Zealand

Bangladesh and India
Winner: India

England and Australia
Winner: Australia

Pakistan and South Africa
Winner: South Africa

New Zealand and Netherlands
Winner: Netherlands

Afghanistan and Zimbabwe
Winner: Zimbabwe

Sri Lanka and Namibia
Winner: Namibia

Pakistan and India
Winner: India

Australia and New Zealand
Winner: New Zealand

Bangladesh and South Africa
Winner: South Africa

England and Sri Lanka
Winner: Sri Lanka

Afghanistan and India
Winner: India

Namibia and Netherlands
Winner: Netherlands

Zimbabwe and Pakistan
Winner: Pakistan

Sri Lanka and Australia
Winner: Australia

South Africa and India
Winner: India

England and New Zealand
Winner: New Zealand

Bangladesh and Afghanistan
Winner: Afghanistan

Netherlands and

In [55]:
#list of tuples before
semi = [
    ("India","South Africa"), ( "Australia", "New Zealand")
    ]


In [56]:
def clean_and_predict(matches, ranking, final, svm):

    import pandas as pd

    ranking["Team"] = ranking["Team"].str.strip()

    pred_set = []
    backup = []

    # -------------------------
    # Build dataset safely
    # -------------------------
    for match in matches:

        team1 = match[0].strip()
        team2 = match[1].strip()

        r1 = ranking.loc[ranking["Team"] == team1, "Rank"].values
        r2 = ranking.loc[ranking["Team"] == team2, "Rank"].values

        if len(r1) == 0 or len(r2) == 0:
            continue

        if r1[0] < r2[0]:
            pred_set.append({"Team1": team1, "Team2": team2})
            backup.append((team1, team2))
        else:
            pred_set.append({"Team1": team2, "Team2": team1})
            backup.append((team2, team1))

    pred_set = pd.DataFrame(pred_set)
    backup = pd.DataFrame(backup, columns=["Team1", "Team2"])

    # -------------------------
    # One-hot encoding
    # -------------------------
    pred_set = pd.get_dummies(pred_set, columns=["Team1", "Team2"])

    # -------------------------
    # Align with training data
    # -------------------------
    pred_set = pred_set.reindex(columns=final.columns, fill_value=0)

    if "Winner" in pred_set.columns:
        pred_set = pred_set.drop(["Winner"], axis=1)

    # -------------------------
    # Predict
    # -------------------------
    predictions = svm.predict(pred_set)

    # -------------------------
    # Output
    # -------------------------
    for i in range(len(pred_set)):

        t1 = backup.iloc[i, 0]
        t2 = backup.iloc[i, 1]

        print(f"{t1} vs {t2}")

        if predictions[i] == 1:
            print("Winner:", t1)
        else:
            print("Winner:", t2)

        print("")

In [57]:
clean_and_predict(semi, ranking, final, svm)

India vs South Africa
Winner: South Africa

New Zealand vs Australia
Winner: Australia



In [68]:
finals =[('South Africa', 'Australia')]

In [69]:
clean_and_predict(finals, ranking, final, svm)

Australia vs South Africa
Winner: South Africa

